Load the Data

In [1900]:
import pandas as pd
import numpy as np

df=pd.read_csv('customer_signups.csv')
df

,customer_id,name,email,signup_date,source,region,plan_selected,marketing_opt_in,age,gender
0,CUST00000,Joshua Bryant,NaN,NaN,Instagram,NaN,basic,No,34,Female
1,CUST00001,Nicole Stewart,nicole1@example.com,02-01-24,LinkedIn,West,basic,Yes,29,Male
2,CUST00002,Rachel Allen,rachel2@example.com,03-01-24,Google,North,PREMIUM,Yes,34,Non-Binary
3,CUST00003,Zachary Sanchez,zachary3@mailhub.org,04-01-24,YouTube,NaN,Pro,No,40,Male
4,CUST00004,NaN,matthew4@mailhub.org,05-01-24,LinkedIn,West,Premium,No,25,Other
...,...,...,...,...,...,...,...,...,...,...
295,CUST00295,Gary Smith,gary95@example.com,22-10-24,Google,West,PREMIUM,Yes,40,NaN
296,CUST00296,Anthony Roberts,anthony96@mailhub.org,23-10-24,Google,Central,Basic,Yes,25,Female
297,CUST00297,Timothy Mclaughlin,NaN,24-10-24,Instagram,West,Basic,Yes,60,NaN
298,CUST00298,Justin Mcintyre,justin98@mailhub.org,25-10-24,YouTube,South,Premium,No,53,male


In [1901]:
dfs=pd.read_csv('support_tickets.csv')
dfs


,ticket_id,customer_id,ticket_date,issue_type,resolved
0,TKT0000-1,CUST00203,2024-08-17,Billing,Yes
1,TKT0000-2,CUST00203,2024-07-22,Technical Error,Yes
2,TKT0000-3,CUST00203,2024-07-22,Other,Yes
3,TKT0001-1,CUST00266,2024-09-26,Account Setup,Yes
4,TKT0001-2,CUST00266,2024-10-09,Technical Error,No
...,...,...,...,...,...
118,TKT0057-1,CUST00092,2024-02-11,Billing,Yes
119,TKT0058-1,CUST00192,2024-12-02,Billing,Yes
120,TKT0058-2,CUST00192,2024-11-28,Account Setup,Yes
121,TKT0058-3,CUST00192,2024-11-18,Login Issue,Yes


In [1902]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   customer_id       298 non-null    str  
 1   name              291 non-null    str  
 2   email             266 non-null    str  
 3   signup_date       298 non-null    str  
 4   source            291 non-null    str  
 5   region            270 non-null    str  
 6   plan_selected     292 non-null    str  
 7   marketing_opt_in  290 non-null    str  
 8   age               288 non-null    str  
 9   gender            292 non-null    str  
dtypes: str(10)
memory usage: 23.6 KB


In [1903]:
df.describe()

,customer_id,name,email,signup_date,source,region,plan_selected,marketing_opt_in,age,gender
count,298,291,266,298,291,270,292,290,288,292
unique,298,291,265,295,7,5,8,3,11,7
top,CUST00000,Joshua Bryant,lisa11@mailhub.org,not a date,YouTube,North,Premium,No,40,Other
freq,1,1,2,4,58,65,57,156,50,59


In [1904]:
dfs.info()

<class 'pandas.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   ticket_id    123 non-null    str  
 1   customer_id  123 non-null    str  
 2   ticket_date  123 non-null    str  
 3   issue_type   123 non-null    str  
 4   resolved     123 non-null    str  
dtypes: str(5)
memory usage: 4.9 KB


Identify Missing Values

In [1905]:
df.isnull().sum()

customer_id          2
name                 9
email               34
signup_date          2
source               9
region              30
plan_selected        8
marketing_opt_in    10
age                 12
gender               8
dtype: int64

In [1906]:
dfs.isnull().sum()

ticket_id      0
customer_id    0
ticket_date    0
issue_type     0
resolved       0
dtype: int64

% of missing values

In [1907]:
missing_values = ((df.isnull().mean()) * 100).round(2)
print(missing_values)

customer_id          0.67
name                 3.00
email               11.33
signup_date          0.67
source               3.00
region              10.00
plan_selected        2.67
marketing_opt_in     3.33
age                  4.00
gender               2.67
dtype: float64


Which region shows signs of missing or incomplete data?

In [1908]:
col=['plan_selected','marketing_opt_in' ]
missing_data=df.groupby('region',dropna=False)[col].apply(lambda x: x.isnull().sum())
print(missing_data)

         plan_selected  marketing_opt_in
region                                  
Central              0                 2
East                 2                 3
North                3                 2
South                0                 2
West                 2                 1
NaN                  1                 0


In [1909]:
q2 = (
    df.groupby("region", dropna=False)
    .apply(lambda g: pd.Series({
        "customers":     len(g),
        "missing_email": g["email"].isna().sum(),
        "missing_age":   g["age"].isna().sum(),
        "missing_opt_in":g["marketing_opt_in"].isna().sum(),
        "missing_plan":  g["plan_selected"].isna().sum(),
    }))
    .reset_index()
)

q2["total_missing"] = q2[["missing_email", "missing_age",
                           "missing_opt_in", "missing_plan"]].sum(axis=1)

# Missing % = total missing fields / (customers × 4 fields) × 100
q2["missing_pct"] = (
    q2["total_missing"] / (q2["customers"] * 4) * 100
).round(1)

print(q2.sort_values("missing_pct", ascending=False).to_string(index=False))
print(f"\nRows with NO region recorded: {df['region'].isna().sum()}")

 region  customers  missing_email  missing_age  missing_opt_in  missing_plan  total_missing  missing_pct
  North         65             10            3               2             3             18          6.9
Central         39              5            2               2             0              9          5.8
   East         61              7            2               3             2             14          5.7
    NaN         30              4            1               0             1              6          5.0
   West         46              4            2               1             2              9          4.9
  South         59              4            2               2             0              8          3.4

Rows with NO region recorded: 30


In [1910]:
dfs.isnull().sum()

ticket_id      0
customer_id    0
ticket_date    0
issue_type     0
resolved       0
dtype: int64

In [1911]:
df.isnull().aggregate([ 'sum'])

,customer_id,name,email,signup_date,source,region,plan_selected,marketing_opt_in,age,gender
sum,2,9,34,2,9,30,8,10,12,8


Check Data Types

In [1912]:
df.dtypes

customer_id         str
name                str
email               str
signup_date         str
source              str
region              str
plan_selected       str
marketing_opt_in    str
age                 str
gender              str
dtype: object

In [1913]:
dfs.dtypes

ticket_id      str
customer_id    str
ticket_date    str
issue_type     str
resolved       str
dtype: object

Convert signup_date to Datetime

In [1914]:
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
df['age']=pd.to_numeric(df['age'],errors='coerce').round(0).astype('Int64')
df

/var/folders/ny/mvjvnvb130bdw8728wx5q__w0000gn/T/ipykernel_28827/1989346467.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')


,customer_id,name,email,signup_date,source,region,plan_selected,marketing_opt_in,age,gender
0,CUST00000,Joshua Bryant,NaN,NaT,Instagram,NaN,basic,No,34,Female
1,CUST00001,Nicole Stewart,nicole1@example.com,2024-02-01,LinkedIn,West,basic,Yes,29,Male
2,CUST00002,Rachel Allen,rachel2@example.com,2024-03-01,Google,North,PREMIUM,Yes,34,Non-Binary
3,CUST00003,Zachary Sanchez,zachary3@mailhub.org,2024-04-01,YouTube,NaN,Pro,No,40,Male
4,CUST00004,NaN,matthew4@mailhub.org,2024-05-01,LinkedIn,West,Premium,No,25,Other
...,...,...,...,...,...,...,...,...,...,...
295,CUST00295,Gary Smith,gary95@example.com,2024-10-22,Google,West,PREMIUM,Yes,40,NaN
296,CUST00296,Anthony Roberts,anthony96@mailhub.org,2024-10-23,Google,Central,Basic,Yes,25,Female
297,CUST00297,Timothy Mclaughlin,NaN,2024-10-24,Instagram,West,Basic,Yes,60,NaN
298,CUST00298,Justin Mcintyre,justin98@mailhub.org,2024-10-25,YouTube,South,Premium,No,53,male


In [1915]:
df.dtypes

customer_id                    str
name                           str
email                          str
signup_date         datetime64[us]
source                         str
region                         str
plan_selected                  str
marketing_opt_in               str
age                          Int64
gender                         str
dtype: object

In [1916]:
dfs['ticket_date']=pd.to_datetime(dfs['ticket_date'],errors='coerce')
dfs

,ticket_id,customer_id,ticket_date,issue_type,resolved
0,TKT0000-1,CUST00203,2024-08-17,Billing,Yes
1,TKT0000-2,CUST00203,2024-07-22,Technical Error,Yes
2,TKT0000-3,CUST00203,2024-07-22,Other,Yes
3,TKT0001-1,CUST00266,2024-09-26,Account Setup,Yes
4,TKT0001-2,CUST00266,2024-10-09,Technical Error,No
...,...,...,...,...,...
118,TKT0057-1,CUST00092,2024-02-11,Billing,Yes
119,TKT0058-1,CUST00192,2024-12-02,Billing,Yes
120,TKT0058-2,CUST00192,2024-11-28,Account Setup,Yes
121,TKT0058-3,CUST00192,2024-11-18,Login Issue,Yes


Standardisation of Text Values

In [1917]:
df=df.replace({'plan_selected':{'Basic':'Basic','basic':'Basic','PREMIUM':'Premium','premium':'Premium','prem':'Premium','PRO':'Pro'},
                                'gender':{'Male':'Male','male':'Male','Female':'Female','female':'Female',"Non-Binary": "Non-Binary",
    "Other":"Other"}})

df                                          


,customer_id,name,email,signup_date,source,region,plan_selected,marketing_opt_in,age,gender
0,CUST00000,Joshua Bryant,NaN,NaT,Instagram,NaN,Basic,No,34,Female
1,CUST00001,Nicole Stewart,nicole1@example.com,2024-02-01,LinkedIn,West,Basic,Yes,29,Male
2,CUST00002,Rachel Allen,rachel2@example.com,2024-03-01,Google,North,Premium,Yes,34,Non-Binary
3,CUST00003,Zachary Sanchez,zachary3@mailhub.org,2024-04-01,YouTube,NaN,Pro,No,40,Male
4,CUST00004,NaN,matthew4@mailhub.org,2024-05-01,LinkedIn,West,Premium,No,25,Other
...,...,...,...,...,...,...,...,...,...,...
295,CUST00295,Gary Smith,gary95@example.com,2024-10-22,Google,West,Premium,Yes,40,NaN
296,CUST00296,Anthony Roberts,anthony96@mailhub.org,2024-10-23,Google,Central,Basic,Yes,25,Female
297,CUST00297,Timothy Mclaughlin,NaN,2024-10-24,Instagram,West,Basic,Yes,60,NaN
298,CUST00298,Justin Mcintyre,justin98@mailhub.org,2024-10-25,YouTube,South,Premium,No,53,Male


Dropping Title

In [1918]:
df["name"] = df["name"].str.replace(
    r'^(Mr\.?|Mrs\.?|Ms\.?|Dr\.?)\s+', 
    '', 
    regex=True,
    flags=0
)
df

,customer_id,name,email,signup_date,source,region,plan_selected,marketing_opt_in,age,gender
0,CUST00000,Joshua Bryant,NaN,NaT,Instagram,NaN,Basic,No,34,Female
1,CUST00001,Nicole Stewart,nicole1@example.com,2024-02-01,LinkedIn,West,Basic,Yes,29,Male
2,CUST00002,Rachel Allen,rachel2@example.com,2024-03-01,Google,North,Premium,Yes,34,Non-Binary
3,CUST00003,Zachary Sanchez,zachary3@mailhub.org,2024-04-01,YouTube,NaN,Pro,No,40,Male
4,CUST00004,NaN,matthew4@mailhub.org,2024-05-01,LinkedIn,West,Premium,No,25,Other
...,...,...,...,...,...,...,...,...,...,...
295,CUST00295,Gary Smith,gary95@example.com,2024-10-22,Google,West,Premium,Yes,40,NaN
296,CUST00296,Anthony Roberts,anthony96@mailhub.org,2024-10-23,Google,Central,Basic,Yes,25,Female
297,CUST00297,Timothy Mclaughlin,NaN,2024-10-24,Instagram,West,Basic,Yes,60,NaN
298,CUST00298,Justin Mcintyre,justin98@mailhub.org,2024-10-25,YouTube,South,Premium,No,53,Male


Count duplicates

In [1919]:
count=df['customer_id'].duplicated().sum()
print(count)

1


Remove Duplicate Rows

In [1920]:

df=df.drop_duplicates(subset='customer_id', keep='first')
df

,customer_id,name,email,signup_date,source,region,plan_selected,marketing_opt_in,age,gender
0,CUST00000,Joshua Bryant,NaN,NaT,Instagram,NaN,Basic,No,34,Female
1,CUST00001,Nicole Stewart,nicole1@example.com,2024-02-01,LinkedIn,West,Basic,Yes,29,Male
2,CUST00002,Rachel Allen,rachel2@example.com,2024-03-01,Google,North,Premium,Yes,34,Non-Binary
3,CUST00003,Zachary Sanchez,zachary3@mailhub.org,2024-04-01,YouTube,NaN,Pro,No,40,Male
4,CUST00004,NaN,matthew4@mailhub.org,2024-05-01,LinkedIn,West,Premium,No,25,Other
...,...,...,...,...,...,...,...,...,...,...
295,CUST00295,Gary Smith,gary95@example.com,2024-10-22,Google,West,Premium,Yes,40,NaN
296,CUST00296,Anthony Roberts,anthony96@mailhub.org,2024-10-23,Google,Central,Basic,Yes,25,Female
297,CUST00297,Timothy Mclaughlin,NaN,2024-10-24,Instagram,West,Basic,Yes,60,NaN
298,CUST00298,Justin Mcintyre,justin98@mailhub.org,2024-10-25,YouTube,South,Premium,No,53,Male


In [1921]:
df.dropna(subset=['customer_id'], inplace=True)
df

,customer_id,name,email,signup_date,source,region,plan_selected,marketing_opt_in,age,gender
0,CUST00000,Joshua Bryant,NaN,NaT,Instagram,NaN,Basic,No,34,Female
1,CUST00001,Nicole Stewart,nicole1@example.com,2024-02-01,LinkedIn,West,Basic,Yes,29,Male
2,CUST00002,Rachel Allen,rachel2@example.com,2024-03-01,Google,North,Premium,Yes,34,Non-Binary
3,CUST00003,Zachary Sanchez,zachary3@mailhub.org,2024-04-01,YouTube,NaN,Pro,No,40,Male
4,CUST00004,NaN,matthew4@mailhub.org,2024-05-01,LinkedIn,West,Premium,No,25,Other
...,...,...,...,...,...,...,...,...,...,...
295,CUST00295,Gary Smith,gary95@example.com,2024-10-22,Google,West,Premium,Yes,40,NaN
296,CUST00296,Anthony Roberts,anthony96@mailhub.org,2024-10-23,Google,Central,Basic,Yes,25,Female
297,CUST00297,Timothy Mclaughlin,NaN,2024-10-24,Instagram,West,Basic,Yes,60,NaN
298,CUST00298,Justin Mcintyre,justin98@mailhub.org,2024-10-25,YouTube,South,Premium,No,53,Male


Handle Missing Values

In [1952]:
df['region']=df['region'].fillna('Unknown')
df['source']=(df['source'].replace({ "??": 'Unknown'}).fillna('Unknown'))
df['age']=df['age'].replace({ "Nil": 'NaN'}).fillna(df['age'].median())
df['gender']=(df['gender'].replace({' ': 'Other', '123': 'Other', 123: 'Other','Female':'Female','Male':'Male','FEMALE':'Female','MALE':'Male'}).fillna('Other'))
df['name']=df['name'].fillna('Unknown')
df['signup_date']=df['signup_date'].fillna(df['signup_date'].median())
df['marketing_opt_in']=(df['marketing_opt_in'].replace({' Nil': 'NAN'}).fillna('Unknown'))
df['plan_selected']=(df['plan_selected'].replace({'Unknown': 'UnknownPlan'}).fillna('UnknownPlan'))





In [1953]:
df.isnull().sum()

customer_id         0
name                0
email               0
signup_date         0
source              0
region              0
plan_selected       0
marketing_opt_in    0
age                 0
gender              0
first_name          0
last_name           9
age_group           0
weeks               0
month               0
dtype: int64

In [1954]:
df['plan_selected'].unique()

<StringArray>
['Basic', 'Premium', 'Pro', 'UnknownPlan']
Length: 4, dtype: str

In [1955]:
df[["first_name", "last_name"]] = df["name"].str.strip().str.split(" ", n = 1, expand = True)
df

,customer_id,name,email,signup_date,source,region,plan_selected,marketing_opt_in,age,gender,first_name,last_name,age_group,weeks,month
0,CUST00000,Joshua Bryant,Joshua0@example.com,2024-06-08 12:00:00,Instagram,Unknown,Basic,No,34,Female,Joshua,Bryant,Youth_Adult,2024-06-03/2024-06-09,2024-06
1,CUST00001,Nicole Stewart,nicole1@example.com,2024-02-01 00:00:00,LinkedIn,West,Basic,Yes,29,Male,Nicole,Stewart,Youth_Adult,2024-01-29/2024-02-04,2024-02
2,CUST00002,Rachel Allen,rachel2@example.com,2024-03-01 00:00:00,Google,North,Premium,Yes,34,Non-Binary,Rachel,Allen,Youth_Adult,2024-02-26/2024-03-03,2024-03
3,CUST00003,Zachary Sanchez,zachary3@mailhub.org,2024-04-01 00:00:00,YouTube,Unknown,Pro,No,40,Male,Zachary,Sanchez,Adult,2024-04-01/2024-04-07,2024-04
4,CUST00004,Unknown,matthew4@mailhub.org,2024-05-01 00:00:00,LinkedIn,West,Premium,No,25,Other,Unknown,NaN,Youth,2024-04-29/2024-05-05,2024-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,CUST00295,Gary Smith,gary95@example.com,2024-10-22 00:00:00,Google,West,Premium,Yes,40,Other,Gary,Smith,Adult,2024-10-21/2024-10-27,2024-10
296,CUST00296,Anthony Roberts,anthony96@mailhub.org,2024-10-23 00:00:00,Google,Central,Basic,Yes,25,Female,Anthony,Roberts,Youth,2024-10-21/2024-10-27,2024-10
297,CUST00297,Timothy Mclaughlin,Timothy297@example.com,2024-10-24 00:00:00,Instagram,West,Basic,Yes,60,Other,Timothy,Mclaughlin,Older,2024-10-21/2024-10-27,2024-10
298,CUST00298,Justin Mcintyre,justin98@mailhub.org,2024-10-25 00:00:00,YouTube,South,Premium,No,53,Male,Justin,Mcintyre,Older,2024-10-21/2024-10-27,2024-10


In [1956]:
df["email"] = df.apply(
    lambda row: f"{row['first_name']}{row.name}@example.com"
    if pd.isnull(row["email"])
    else row["email"],
    axis=1
)

In [1957]:
df.groupby('marketing_opt_in')['customer_id'].count().reset_index(name='count')

,marketing_opt_in,count
0,No,156
1,Unknown,10
2,Yes,131


In [1958]:
df

,customer_id,name,email,signup_date,source,region,plan_selected,marketing_opt_in,age,gender,first_name,last_name,age_group,weeks,month
0,CUST00000,Joshua Bryant,Joshua0@example.com,2024-06-08 12:00:00,Instagram,Unknown,Basic,No,34,Female,Joshua,Bryant,Youth_Adult,2024-06-03/2024-06-09,2024-06
1,CUST00001,Nicole Stewart,nicole1@example.com,2024-02-01 00:00:00,LinkedIn,West,Basic,Yes,29,Male,Nicole,Stewart,Youth_Adult,2024-01-29/2024-02-04,2024-02
2,CUST00002,Rachel Allen,rachel2@example.com,2024-03-01 00:00:00,Google,North,Premium,Yes,34,Non-Binary,Rachel,Allen,Youth_Adult,2024-02-26/2024-03-03,2024-03
3,CUST00003,Zachary Sanchez,zachary3@mailhub.org,2024-04-01 00:00:00,YouTube,Unknown,Pro,No,40,Male,Zachary,Sanchez,Adult,2024-04-01/2024-04-07,2024-04
4,CUST00004,Unknown,matthew4@mailhub.org,2024-05-01 00:00:00,LinkedIn,West,Premium,No,25,Other,Unknown,NaN,Youth,2024-04-29/2024-05-05,2024-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,CUST00295,Gary Smith,gary95@example.com,2024-10-22 00:00:00,Google,West,Premium,Yes,40,Other,Gary,Smith,Adult,2024-10-21/2024-10-27,2024-10
296,CUST00296,Anthony Roberts,anthony96@mailhub.org,2024-10-23 00:00:00,Google,Central,Basic,Yes,25,Female,Anthony,Roberts,Youth,2024-10-21/2024-10-27,2024-10
297,CUST00297,Timothy Mclaughlin,Timothy297@example.com,2024-10-24 00:00:00,Instagram,West,Basic,Yes,60,Other,Timothy,Mclaughlin,Older,2024-10-21/2024-10-27,2024-10
298,CUST00298,Justin Mcintyre,justin98@mailhub.org,2024-10-25 00:00:00,YouTube,South,Premium,No,53,Male,Justin,Mcintyre,Older,2024-10-21/2024-10-27,2024-10


Age Group

In [1929]:
df['age_group'] = pd.cut(df['age'], bins=[0, 18, 25, 35, 50, np.inf], labels=['Young', 'Youth', 'Youth_Adult', 'Adult', 'Older'])


Aggregations

Sign-ups per week 

In [1959]:
df['weeks'] = df['signup_date'].dt.to_period('W')
df

,customer_id,name,email,signup_date,source,region,plan_selected,marketing_opt_in,age,gender,first_name,last_name,age_group,weeks,month
0,CUST00000,Joshua Bryant,Joshua0@example.com,2024-06-08 12:00:00,Instagram,Unknown,Basic,No,34,Female,Joshua,Bryant,Youth_Adult,2024-06-03/2024-06-09,2024-06
1,CUST00001,Nicole Stewart,nicole1@example.com,2024-02-01 00:00:00,LinkedIn,West,Basic,Yes,29,Male,Nicole,Stewart,Youth_Adult,2024-01-29/2024-02-04,2024-02
2,CUST00002,Rachel Allen,rachel2@example.com,2024-03-01 00:00:00,Google,North,Premium,Yes,34,Non-Binary,Rachel,Allen,Youth_Adult,2024-02-26/2024-03-03,2024-03
3,CUST00003,Zachary Sanchez,zachary3@mailhub.org,2024-04-01 00:00:00,YouTube,Unknown,Pro,No,40,Male,Zachary,Sanchez,Adult,2024-04-01/2024-04-07,2024-04
4,CUST00004,Unknown,matthew4@mailhub.org,2024-05-01 00:00:00,LinkedIn,West,Premium,No,25,Other,Unknown,NaN,Youth,2024-04-29/2024-05-05,2024-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,CUST00295,Gary Smith,gary95@example.com,2024-10-22 00:00:00,Google,West,Premium,Yes,40,Other,Gary,Smith,Adult,2024-10-21/2024-10-27,2024-10
296,CUST00296,Anthony Roberts,anthony96@mailhub.org,2024-10-23 00:00:00,Google,Central,Basic,Yes,25,Female,Anthony,Roberts,Youth,2024-10-21/2024-10-27,2024-10
297,CUST00297,Timothy Mclaughlin,Timothy297@example.com,2024-10-24 00:00:00,Instagram,West,Basic,Yes,60,Other,Timothy,Mclaughlin,Older,2024-10-21/2024-10-27,2024-10
298,CUST00298,Justin Mcintyre,justin98@mailhub.org,2024-10-25 00:00:00,YouTube,South,Premium,No,53,Male,Justin,Mcintyre,Older,2024-10-21/2024-10-27,2024-10


In [1960]:
weekly = df.groupby('weeks')['customer_id'].count().reset_index(name='signups per week')
weekly


,weeks,signups per week
0,2024-01-01/2024-01-07,6
1,2024-01-08/2024-01-14,5
2,2024-01-15/2024-01-21,7
3,2024-01-22/2024-01-28,7
4,2024-01-29/2024-02-04,8
5,2024-02-05/2024-02-11,6
6,2024-02-12/2024-02-18,6
7,2024-02-19/2024-02-25,7
8,2024-02-26/2024-03-03,7
9,2024-03-04/2024-03-10,7


Sign-ups by source, region, and plan_selected

In [1961]:
df.groupby(['region', 'source', 'plan_selected']).size().reset_index(name='signups per group')

,region,source,plan_selected,signups per group
0,Central,Facebook,Premium,4
1,Central,Facebook,Pro,3
2,Central,Google,Basic,2
3,Central,Google,Premium,2
4,Central,Google,Pro,6
...,...,...,...,...
114,West,Unknown,Pro,2
115,West,YouTube,Basic,1
116,West,YouTube,Premium,5
117,West,YouTube,Pro,1


In [1962]:
df.groupby('source').size().sort_values(ascending=False).reset_index(name='signups per source')

,source,signups per source
0,YouTube,58
1,Google,50
2,Referral,49
3,Instagram,48
4,Facebook,40
5,LinkedIn,37
6,Unknown,15


Marketing opt-in counts by gender

In [1963]:
df.groupby('gender',dropna=False)['marketing_opt_in'].count().reset_index(name='count')

,gender,count
0,Female,92
1,Male,90
2,Non-Binary,42
3,Other,73


In [1964]:
gender_table=pd.crosstab(df['gender'],df['marketing_opt_in'], margins=True, margins_name='Total')
gender_table

marketing_opt_in,No,Unknown,Yes,Total
gender,,,,
Female,47,1,44,92
Male,50,3,37,90
Non-Binary,20,3,19,42
Other,39,3,31,73
Total,156,10,131,297


Age summary: min, max, mean, median, null count

checking the uniqueness of the age

In [1936]:
sorted(df['age'].dropna().unique())

[np.int64(21),
 np.int64(25),
 np.int64(29),
 np.int64(34),
 np.int64(40),
 np.int64(47),
 np.int64(53),
 np.int64(60),
 np.int64(206)]

206 is not real age. remove impossible age

In [1965]:
df['age']=df['age'].where(df['age'].between(18, 100))

In [1966]:
df["age"].agg(count="count", null_count=lambda x: x.isna().sum(), minimum="min", maximum="max", mean="mean", median="median", std_dev="std").reset_index(name="summary").astype({"summary": "int64"})


,index,summary
0,count,297
1,null_count,0
2,minimum,21
3,maximum,60
4,mean,35
5,median,34
6,std_dev,10


These Business 
Answer the following using your analysis. Write clear, concise answers in your PDF report:

2. Which region shows signs of missing or incomplete data?
3. Are older users more or less likely to opt in to marketing?
4. Which plan is most commonly selected, and by which age group?
5. (Optional) Which plan’s users are most likely to contact support?

1. Which acquisition source brought in the most users last month?

In [1967]:
df['month']=df['signup_date'].dt.to_period('M')
prv_month = df['month'].max()-1
last_month_signups = df[df['month'] == prv_month]
last_month_aquision=last_month_signups['source'].value_counts().reset_index(name='signups per source')
last_month_aquision

,source,signups per source
0,Google,3
1,Instagram,2
2,Unknown,2
3,Referral,1
4,Facebook,1
5,LinkedIn,1


In [1968]:
last_date   = df['signup_date'].max()
month_start = last_date - pd.DateOffset(months=1)
last_month  = df[df['signup_date'] >= month_start]
 
q1 = (
    last_month['source']
    .value_counts(dropna=False)
    .reset_index()
    .assign(share_pct=lambda x: (x['count'] / x['count'].sum() * 100).round(1))
)
print(f'Window: {month_start.date()} -> {last_date.date()}')
print(q1.to_string(index=False))


Window: 2024-11-10 -> 2024-12-10
   source  count  share_pct
Instagram      4       36.4
 Referral      2       18.2
  YouTube      2       18.2
 Facebook      1        9.1
 LinkedIn      1        9.1
  Unknown      1        9.1


3. Are older users more or less likely to opt in to marketing?

In [1969]:
df['marketing_opt_in']=df['marketing_opt_in'].replace('Nil',np.nan)
df.dropna(subset=['marketing_opt_in'], inplace=True)

In [1970]:
users=df.groupby(['age_group','marketing_opt_in' ]).size().reset_index(name='count')


user=users.pivot(index='age_group', columns='marketing_opt_in', values='count').fillna(0)
user
user['total'] = user.sum(axis=1)
user['percentage_opt_in'] = ((user['Yes'] / user['total']) * 100).round(2).astype(str) + '%'

user

marketing_opt_in,No,Unknown,Yes,total,percentage_opt_in
age_group,,,,,
Youth,41,4,30,75,40.0%
Youth_Adult,57,3,48,108,44.44%
Adult,35,2,35,72,48.61%
Older,23,1,18,42,42.86%


4. Which plan is most commonly selected, and by which age group?

In [1971]:
selected=df.groupby( ['plan_selected','age_group']).size().reset_index(name='count')
plan=selected.pivot(index='age_group', columns='plan_selected', values='count').fillna(0).astype(int)
plan['total'] = plan.sum(axis=1)

plan

plan_selected,Basic,Premium,Pro,UnknownPlan,total
age_group,,,,,
Youth,28,22,23,2,75
Youth_Adult,33,35,35,5,108
Adult,16,32,19,5,72
Older,15,10,15,2,42


In [1972]:
table=df.merge(dfs, on='customer_id', how='inner')
table

,customer_id,name,email,signup_date,source,region,plan_selected,marketing_opt_in,age,gender,first_name,last_name,age_group,weeks,month,ticket_id,ticket_date,issue_type,resolved
0,CUST00005,John Gonzales,john5@mailhub.org,2024-06-01,Facebook,South,Premium,No,34,Other,John,Gonzales,Youth_Adult,2024-05-27/2024-06-02,2024-06,TKT0008-1,2024-06-04,Other,Yes
1,CUST00007,Michael Bailey,michael7@mailhub.org,2024-08-01,YouTube,Central,Pro,Yes,60,Other,Michael,Bailey,Older,2024-07-29/2024-08-04,2024-08,TKT0036-1,2024-08-07,Billing,Yes
2,CUST00007,Michael Bailey,michael7@mailhub.org,2024-08-01,YouTube,Central,Pro,Yes,60,Other,Michael,Bailey,Older,2024-07-29/2024-08-04,2024-08,TKT0036-2,2024-08-23,Other,Yes
3,CUST00009,Cindy Anderson,Cindy9@example.com,2024-10-01,Google,East,Premium,No,29,Female,Cindy,Anderson,Youth_Adult,2024-09-30/2024-10-06,2024-10,TKT0003-1,2024-10-03,Technical Error,No
4,CUST00017,Patty Paul,patty17@inboxmail.net,2024-01-18,YouTube,East,Pro,No,53,Non-Binary,Patty,Paul,Older,2024-01-15/2024-01-21,2024-01,TKT0030-1,2024-02-03,Other,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,CUST00289,Daniel Dunn,daniel89@example.com,2024-10-16,Referral,West,Basic,No,34,Male,Daniel,Dunn,Youth_Adult,2024-10-14/2024-10-20,2024-10,TKT0016-1,2024-11-10,Billing,Yes
119,CUST00295,Gary Smith,gary95@example.com,2024-10-22,Google,West,Premium,Yes,40,Other,Gary,Smith,Adult,2024-10-21/2024-10-27,2024-10,TKT0027-1,2024-11-01,Other,Yes
120,CUST00297,Timothy Mclaughlin,Timothy297@example.com,2024-10-24,Instagram,West,Basic,Yes,60,Other,Timothy,Mclaughlin,Older,2024-10-21/2024-10-27,2024-10,TKT0054-1,2024-11-09,Other,Yes
121,CUST00297,Timothy Mclaughlin,Timothy297@example.com,2024-10-24,Instagram,West,Basic,Yes,60,Other,Timothy,Mclaughlin,Older,2024-10-21/2024-10-27,2024-10,TKT0054-2,2024-11-01,Account Setup,Yes


 Calculate gap in days between signup and each ticket

In [1973]:
table['gap_days']=(table['ticket_date']-table['signup_date']).dt.days
table = table[table['gap_days'] >= 0]

In [1974]:
table['days_group'] = pd.cut(
    table['gap_days'],
    bins=[-1, 14, np.inf],
    labels=['within_14_days', 'After_14_days']
)

In [1975]:
result = (
    table.groupby('days_group')['customer_id']
    .nunique()
    .reset_index(name='customer_count')
)

result

,days_group,customer_count
0,within_14_days,47
1,After_14_days,37


5. (Optional) Which plan’s users are most likely to contact support?

In [1976]:
support=table.groupby(['plan_selected','issue_type']).size().reset_index(name='count')
support
ticket_support=support.pivot(index='plan_selected', columns='issue_type', values='count').fillna(0).astype(int)
ticket_support['total'] = ticket_support.sum(axis=1)
ticket_support.sort_values(by='total', ascending=False)
ticket_support

issue_type,Account Setup,Billing,Login Issue,Other,Technical Error,total
plan_selected,,,,,,
Basic,7,10,11,9,5,42
Premium,6,4,6,6,4,26
Pro,7,11,9,11,9,47
UnknownPlan,0,1,3,1,3,8


 Count how many customers contacted support within 2 weeks of sign-up

In [1977]:
table['days_group'] = pd.cut(table['gap_days'], bins=[-999,-1, 14,  np.inf], labels=['Before_signup', 'within_14_days', 'After_14_days'])
table.groupby('days_group')['customer_id'].nunique().reset_index(name='ticket_count')

,days_group,ticket_count
0,within_14_days,47
1,After_14_days,37


Summarise support activity by plan and region (Group by plan and region)

In [1978]:
s_table=table.groupby(['plan_selected','region'])['ticket_id'].count().reset_index(name='total_tickets').sort_values(by='total_tickets', ascending=False)
s_tables=s_table.pivot(index='plan_selected', columns='region', values='total_tickets').fillna(0).astype(int)
s_tables['total'] = s_tables.sum(axis=1)
s_tables

region,Central,East,North,South,Unknown,West,total
plan_selected,,,,,,,
Basic,2,11,3,14,2,10,42
Premium,6,1,6,2,0,11,26
Pro,10,14,11,3,3,6,47
UnknownPlan,0,2,6,0,0,0,8
